# 设备名称匹配

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 配置matplotlib中文显示
import platform
from rapidfuzz import fuzz, process
from sklearn.cluster import AgglomerativeClustering
system_name = platform.system()

if system_name == 'Windows':
    # Windows系统
    plt.rcParams['font.family'] = ['SimHei']
elif system_name == 'Darwin':
    # macOS系统
    plt.rcParams['font.family'] = ['PingFang HK']
elif system_name == 'Linux':
    # Linux系统（可能需要根据具体发行版进行调整）
    plt.rcParams['font.family'] = ['DejaVu Sans']
else:
    # 其他系统或无法识别系统，默认字体
    plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

# 导入数据

## 油耗表

In [52]:
# import data
file_path = 'Resources/Fuel_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['油耗信息', '设备信息']


In [53]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1.head()

,日期,班次,设备名称,设备编号,油品种类,油品消耗
0,2019-08-04,Night,TEREX TR100#56,HT0056,IC IC,476
1,2019-08-04,Night,TEREX TR100#58,HT0058,IC IC,462
2,2019-08-04,Night,TEREX TR100#59,HT0059,IC IC,451
3,2019-08-04,Night,TEREX TR100#60,HT0060,IC IC,454
4,2019-08-04,Night,TEREX TR100#61,HT0061,IC IC,502


In [54]:
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
df_sheet1['设备编号'] = df_sheet1['设备编号'].str.strip()
df_sheet1.drop_duplicates(inplace=True)

In [55]:
match_table = pd.DataFrame()
match_table['设备名称'] = df_sheet1['设备名称']
match_table['设备编号'] = df_sheet1['设备编号']
match_table['公司'] = np.nan
match_table['来源'] = f"Fuel_{xlsx.sheet_names[0]}"
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
219252,AS4006 LP#0075,LP75,NaN,Fuel_油耗信息
219642,AS4006 LP#0074,LP74,NaN,Fuel_油耗信息
219644,4931ХОҮ/ LV#1232 /JMC / JX1033/,LV#1232,NaN,Fuel_油耗信息
220201,ZTC250A ME#136,ME#136,NaN,Fuel_油耗信息


In [56]:
df_sheet1 = pd.read_excel(file_path, sheet_name=1)
df_sheet1.head()

,日期,班次,设备名称,设备编号,发动机小时数开始,发动机小时数结束,运行小时数
0,2019-09-01,Day,TEREX TR100#58,HT0058,301.1,311,9.9
1,2019-09-01,Day,TEREX TR100#64,HT0064,307.7,307.7,0
2,2019-09-01,Day,NHL NTE240 #66,HT0066,64,64,0
3,2019-09-01,Day,NHL NTE240 #67,HT0067,62,62,0
4,2019-09-01,Day,NHL NTE240 #68,HT0068,66,66,0


In [57]:
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
df_sheet1['设备编号'] = df_sheet1['设备编号'].str.strip()
df_sheet1['来源'] = f"Fuel_{xlsx.sheet_names[1]}"
df_sheet1.drop_duplicates(inplace=True)

In [58]:
match_table = pd.concat([match_table, df_sheet1[['设备名称', '设备编号', '来源']]], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
599234,AS4006 LP#0074,LP74,NaN,Fuel_设备信息
599235,AS4006 LP#0075,LP75,NaN,Fuel_设备信息
599242,4931ХОҮ/ LV#1232 /JMC / JX1033/,LV#1232,NaN,Fuel_设备信息
604475,ZTC250A ME#136,ME#136,NaN,Fuel_设备信息


## 电力表

In [59]:
# import data
file_path = 'Resources/电力_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['电力消耗']


In [60]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1.head()

,日期,设备名称,电力消耗
0,2023-02-01,EX-5600#111,20873.9552
1,2023-02-01,EX-5600#113,26296.6428
2,2023-02-02,EX-5600#111,18047.4620
3,2023-02-02,EX-5600#113,25303.8492
4,2023-02-03,EX-5600#113,26525.0880


In [61]:
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
df_sheet1['设备编号'] = np.nan
df_sheet1['来源'] = f"电力_{xlsx.sheet_names[0]}"
df_sheet1.drop_duplicates(inplace=True)

In [62]:
match_table = pd.concat([match_table, df_sheet1[['设备名称', '设备编号', '来源']]], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
1123,EX-5600#111,NaN,NaN,电力_电力消耗
1124,EX-5600#113,NaN,NaN,电力_电力消耗
1131,EX-5600#122,NaN,NaN,电力_电力消耗
1204,EX-5600#123,NaN,NaN,电力_电力消耗


## 产量

In [63]:
# import data
file_path = 'Resources/产量_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['生产数据', '运行数据']


In [64]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1.head()

,日期,班次,矿卡名称,挖机名称,矿石类型,数量,产量
0,2019-08-17,Day,TEREX TR100 #56,CAT 390D #02,Хө,4.0,128.0
1,2019-08-17,Day,TEREX TR100 #57,CAT 390D #02,Хө,4.0,128.0
2,2019-08-17,Day,TEREX TR100 #58,CAT 390D #02,Хө,3.0,96.0
3,2019-08-17,Day,TEREX TR100 #59,CAT 390D #02,Хө,4.0,128.0
4,2019-08-17,Day,TEREX TR100 #60,CAT 390D #02,Хө,5.0,160.0


In [66]:
df = pd.DataFrame()
df['设备名称'] = df_sheet1['挖机名称'].str.strip().unique()
df['来源'] = f"产量_{xlsx.sheet_names[0]}"
df2 = pd.DataFrame()
df2['设备名称'] = df_sheet1['矿卡名称'].str.strip().unique()
df2['来源'] = f"产量_{xlsx.sheet_names[1]}"
df = pd.concat([df, df2], ignore_index=True)
df.drop_duplicates(inplace=True)
df

,设备名称,来源
0,CAT 390D #02,产量_生产数据
1,PC 1250 #04KH,产量_生产数据
2,LIEBHERR R9350 #06,产量_生产数据
3,CAT 992K #01,产量_生产数据
4,EX5600E-6 #10,产量_生产数据
...,...,...
979,NTE240#1221,产量_运行数据
980,NTE240#1222,产量_运行数据
981,NHL NTE240AC #1221,产量_运行数据
982,NHL NTE240AC #1220,产量_运行数据


In [68]:
match_table = pd.concat([match_table, df[['设备名称', '来源']]], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
2107,NTE240#1221,NaN,NaN,产量_运行数据
2108,NTE240#1222,NaN,NaN,产量_运行数据
2109,NHL NTE240AC #1221,NaN,NaN,产量_运行数据
2110,NHL NTE240AC #1220,NaN,NaN,产量_运行数据


## 效率

In [69]:
# import data
file_path = 'Resources/效率_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['Sheet1']


In [79]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)
df_sheet1.head()

,日期,班次,序号 Д/д,设备种类 Техникийн төрөл,公司 Компани,应运行分钟 Ажиллах мин,应运行小时数 Ажиллах мот цаг,停车/换班/\nСул Зогсолт /Ээлж солигдох /,转移 Шилжих\nхөдөлгөөн,挖机场地推土 /清理墙壁/\nУл талбай түрүүлэх/Ханын цэвэрлэгээ/,...,未计划/故障 Төлөвлөгөөт бус/эвдрэл,待命 Бэлэн байдалд,因天气：大风暴，雨，雪Цаг агаар: Хүчтэй салхи. Бороо.Цас,扬尘：洒水车不足 Тоосжилт: Усны машин хангалтгүй,排队/装水/ Дараалалд зогсох /Ус дүүргэх/,总产量生产运行分钟 Нийт бүтээлтэй ажилласан мин,因电力原因停车。/计划/Цахилгаанаас шалтгаалсан зогсолт ./Төлөвлөсөн/,因电力原因停车。/未计划/Цахилгаанаас шалтгаалсан зогсолт /Төлөвлөгөөт бус/,总产量生产运行小时 Нийт бүтээлтэй ажилласан цаг,注释 Тайлбар
0,2019-08-01,Day,1,LIEBHERR R996B #07,NaN,720,12,60,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,570,NaN,NaN,9.5,NaN
1,2019-08-01,Day,2,HITACHI EX3600 #01KH,NaN,720,12,60,30,30,...,NaN,30,NaN,NaN,NaN,510,NaN,NaN,8.5,NaN
2,2019-08-01,Day,3,HITACHI EX2600E #08,NaN,720,12,60,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,600,NaN,NaN,10,NaN
3,2019-08-01,Day,4,HITACHI EX2600E #03KH,NaN,720,12,60,210,20,...,NaN,NaN,NaN,NaN,NaN,370,NaN,NaN,6.166667,NaN
4,2019-08-01,Day,5,LIEBHERR R9350 #06,NaN,720,12,70,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,580,NaN,NaN,9.666667,NaN


In [80]:
# 查看第4列
df_sheet1['设备名称'] = df_sheet1.iloc[:, 3].str.strip()
df_sheet1['公司'] = df_sheet1.iloc[:, 4].str.strip()
df_sheet1['来源'] = f"效率_{xlsx.sheet_names[0]}"
df = df_sheet1[['设备名称','公司','来源']]
df.drop_duplicates(inplace=True)
df

,设备名称,公司,来源
0,LIEBHERR R996B #07,NaN,效率_Sheet1
1,HITACHI EX3600 #01KH,NaN,效率_Sheet1
2,HITACHI EX2600E #08,NaN,效率_Sheet1
3,HITACHI EX2600E #03KH,NaN,效率_Sheet1
4,LIEBHERR R9350 #06,NaN,效率_Sheet1
...,...,...,...
382524,Terex 60 #1201,Normount,效率_Sheet1
412504,LO#165,Normount,效率_Sheet1
412649,LONKING#165,Normount,效率_Sheet1
413063,设备种类 Техникийн төрөл,公司 Компани,效率_Sheet1


In [81]:
match_table = pd.concat([match_table, df[['设备名称', '来源', '公司']]], ignore_index=True)
match_table.drop_duplicates(inplace=True)
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
3096,Terex 60 #1201,NaN,Normount,效率_Sheet1
3097,LO#165,NaN,Normount,效率_Sheet1
3098,LONKING#165,NaN,Normount,效率_Sheet1
3099,设备种类 Техникийн төрөл,NaN,公司 Компани,效率_Sheet1


In [82]:
match_table.drop_duplicates(inplace=True, subset=['设备名称', '设备编号'])
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
3084,CAT-D9GC #177,NaN,Normount,效率_Sheet1
3097,LO#165,NaN,Normount,效率_Sheet1
3098,LONKING#165,NaN,Normount,效率_Sheet1
3099,设备种类 Техникийн төрөл,NaN,公司 Компани,效率_Sheet1


In [83]:
match_table.drop_duplicates(inplace=True, subset=['设备名称', '公司'])
match_table

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息
...,...,...,...,...
3084,CAT-D9GC #177,NaN,Normount,效率_Sheet1
3097,LO#165,NaN,Normount,效率_Sheet1
3098,LONKING#165,NaN,Normount,效率_Sheet1
3099,设备种类 Техникийн төрөл,NaN,公司 Компани,效率_Sheet1


## 导出

In [84]:
match_table.to_excel('Resources/设备名称匹配清单.xlsx', index=False)

# 匹配

## 导入

In [2]:
# import data
table_to_match_fp = "Resources/设备名称匹配清单.xlsx"
table_to_match = pd.ExcelFile(table_to_match_fp)
print(table_to_match.sheet_names)

['Sheet1']


In [4]:
table_to_match_df = pd.read_excel(table_to_match_fp, sheet_name=0)
table_to_match_df.head()

,设备名称,设备编号,公司,来源
0,TEREX TR100#56,HT0056,NaN,Fuel_油耗信息
1,TEREX TR100#58,HT0058,NaN,Fuel_油耗信息
2,TEREX TR100#59,HT0059,NaN,Fuel_油耗信息
3,TEREX TR100#60,HT0060,NaN,Fuel_油耗信息
4,TEREX TR100#61,HT0061,NaN,Fuel_油耗信息


In [ ]:
# import data
table_to_match_fp = "Resources/设备名称匹配清单.xlsx"
table_to_match = pd.ExcelFile(table_to_match_fp)
print(table_to_match.sheet_names)